# GramText — train the MobileViT STR model (Hindi + English)

Runs on a **free Colab T4 GPU**: *Runtime → Change runtime type → T4 GPU*.

1. Put the `ml` folder (from `C:\MCA\gramtext\ml`, without `data/`) in Google Drive at **MyDrive/gramtext/ml**,
   or upload `ml.zip` when the upload cell asks.
2. Run the cells top to bottom. Checkpoints are saved to Drive, so if Colab disconnects,
   just run everything again — training resumes where it stopped.
3. The last cell downloads `gramtext_str.onnx`. Copy it to `C:\MCA\gramtext\backend\models\`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, zipfile
DRIVE = '/content/drive/MyDrive/gramtext'
os.makedirs(DRIVE, exist_ok=True)
if os.path.isdir(f'{DRIVE}/ml'):
    shutil.copytree(f'{DRIVE}/ml', '/content/ml', dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('data', 'runs', '__pycache__'))
else:
    from google.colab import files
    print('Upload ml.zip')
    name = next(iter(files.upload()))
    zipfile.ZipFile(name).extractall('/content')
    shutil.copytree('/content/ml', f'{DRIVE}/ml', dirs_exist_ok=True)
%cd /content/ml
!ls

In [ ]:
!pip install -q timm uharfbuzz freetype-py onnx onnxruntime

In [ ]:
!python prepare_data.py

**Optional — real data.** Training uses synthetic images by default. If you have real cropped
words (IIIT-ILST Hindi, IndicSTR12, or photos from the field pilot), put them in Drive as
`MyDrive/gramtext/real_train/` and `MyDrive/gramtext/real_val/`, each with a `labels.tsv`
(`image.jpg<TAB>text`). The next cell uses them automatically.

In [ ]:
import os
RUN = f'{DRIVE}/runs/str'
extra = ''
if os.path.exists(f'{DRIVE}/real_train/labels.tsv'): extra += f' --real-train {DRIVE}/real_train'
if os.path.exists(f'{DRIVE}/real_val/labels.tsv'):   extra += f' --real-val {DRIVE}/real_val'
print('extra args:', extra or '(synthetic only)')

## Train
40,000 steps × batch 128 takes roughly 3–5 hours on a T4. Colab may disconnect before that —
re-run all cells and it resumes from the last checkpoint (`--resume`).

In [ ]:
!python train.py --pretrained --resume --steps 40000 --batch 128 --workers 2 --val-every 2000 --out {RUN} {extra}

In [ ]:
import json
print(json.dumps(json.load(open(f'{RUN}/best_metrics.json')), indent=2, ensure_ascii=False))

## Evaluate on a held-out set and export for the backend

In [ ]:
!python eval.py --ckpt {RUN}/best.pt --report {RUN}/eval_synthetic.json
if os.path.exists(f'{DRIVE}/real_val/labels.tsv'):
    !python eval.py --ckpt {RUN}/best.pt --real {DRIVE}/real_val --report {RUN}/eval_real.json

In [ ]:
!python export.py --ckpt {RUN}/best.pt --out-dir {DRIVE} --format onnx
from google.colab import files
files.download(f'{DRIVE}/gramtext_str.onnx')